# Yandex Music

Comparing Moscow and Saint Petersburg is surrounded by many myths. For example:

- Moscow is a megacity shaped by the strict rhythm of the working week.
- Saint Petersburg is the cultural capital with its own distinctive tastes.

Using Yandex Music data, we will compare user behavior in these two cities.

## Study Objective

The goal of this study is to test three hypotheses:

1. User activity depends on the day of the week, and this pattern differs between Moscow and Saint Petersburg.
2. On Monday morning, some genres dominate in Moscow while others dominate in Saint Petersburg. The same pattern is expected on Friday evening, with dominant genres differing by city.
3. Moscow and Saint Petersburg prefer different music genres. Pop music is expected to be more popular in Moscow, while Russian rap is expected to be more popular in Saint Petersburg.

## Study Plan

We will use user behavior data from the file `yandex_music_project.csv`. Since the data quality is unknown, we will first perform a data overview before testing the hypotheses.

We will check the dataset for errors and assess how they may affect the analysis. Then, during preprocessing, we will look for ways to fix the most critical data issues.

The study will include three main stages:

1. Data overview
2. Data preprocessing
3. Hypothesis testing

In [51]:
import pandas as pd

In [2]:
df = pd.read_csv("C:\\Users\\HOME\\PycharmProjects\\2026_classical_ML\\DA_1_MSK_SPb_music_project\\yandex_music_project.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65079 entries, 0 to 65078
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0     userID  65079 non-null  object
 1   Track     63848 non-null  object
 2   artist    57876 non-null  object
 3   genre     63881 non-null  object
 4     City    65079 non-null  object
 5   time      65079 non-null  object
 6   Day       65079 non-null  object
dtypes: object(7)
memory usage: 3.5+ MB


## 1. Data Preprocessing

**1.1 Renaming columns**

In [5]:
df.columns

Index(['  userID', 'Track', 'artist', 'genre', '  City  ', 'time', 'Day'], dtype='object')

We can see that the table contains seven columns. The data type in all columns is `object`.

According to the dataset documentation:

- `userID` — user identifier
- `Track` — track title
- `artist` — artist name
- `genre` — genre name
- `City` — user's city
- `time` — track start time
- `Day` — day of the week

The number of values differs across columns, which means the dataset contains missing values.

We can see several issues in the column names:

- Uppercase and lowercase letters are used inconsistently.
- Some column names contain spaces.
- ` userID` and `City` contain two leading spaces.


In [6]:
df = df.rename(columns = {'  userID':'user_id', 'Track':'track', '  City  ':'city', 'Day':'day'})
print(df)

        user_id                              track            artist  \
0      FFB692EC                  Kamigata To Boots  The Mass Missile   
1      55204538        Delayed Because of Accident  Andreas Rönnberg   
2        20EC38                  Funiculì funiculà       Mario Lanza   
3      A3DD03C9              Dragons in the Sunset        Fire + Ice   
4      E2DC1FAE                        Soul People        Space Echo   
...         ...                                ...               ...   
65074  729CBB09                            My Name            McLean   
65075  D08D4A55  Maybe One Day (feat. Black Spade)       Blu & Exile   
65076  C5E3A0D5                          Jalopiina               NaN   
65077  321D0506                      Freight Train     Chas McDevitt   
65078  3A64EF84          Tell Me Sweet Little Lies      Monica Lopez   

            genre              city      time        day  
0            rock  Saint-Petersburg  20:28:33  Wednesday  
1            rock

In [7]:
df.columns

Index(['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='object')

**1.2 Missing Values**

In [9]:
na_number = df.isna().sum()
print(na_number)

user_id       0
track      1231
artist     7203
genre      1198
city          0
time          0
day           0
dtype: int64


We iterate through the column names and replace missing values with `'unknown'`.

In [10]:
columns_to_replace = [
    'track',
    'artist',
    'genre'
]

for column in columns_to_replace:
    df['track'] = df['track'].fillna('unknown')
    df['artist'] = df['artist'].fillna('unknown')
    df['genre'] = df['genre'].fillna('unknown')
print(df)

        user_id                              track            artist  \
0      FFB692EC                  Kamigata To Boots  The Mass Missile   
1      55204538        Delayed Because of Accident  Andreas Rönnberg   
2        20EC38                  Funiculì funiculà       Mario Lanza   
3      A3DD03C9              Dragons in the Sunset        Fire + Ice   
4      E2DC1FAE                        Soul People        Space Echo   
...         ...                                ...               ...   
65074  729CBB09                            My Name            McLean   
65075  D08D4A55  Maybe One Day (feat. Black Spade)       Blu & Exile   
65076  C5E3A0D5                          Jalopiina           unknown   
65077  321D0506                      Freight Train     Chas McDevitt   
65078  3A64EF84          Tell Me Sweet Little Lies      Monica Lopez   

            genre              city      time        day  
0            rock  Saint-Petersburg  20:28:33  Wednesday  
1            rock

In [11]:
na_number = df.isna().sum()
print(na_number)

user_id    0
track      0
artist     0
genre      0
city       0
time       0
day        0
dtype: int64


**1.3 Dublicates: explicit and implicit**

We count the number of explicit duplicates in the dataset and remove them.

In [13]:
df.duplicated().sum()

3826

In [14]:
df = df.drop_duplicates().reset_index(drop=True)
print(df)

        user_id                              track            artist  \
0      FFB692EC                  Kamigata To Boots  The Mass Missile   
1      55204538        Delayed Because of Accident  Andreas Rönnberg   
2        20EC38                  Funiculì funiculà       Mario Lanza   
3      A3DD03C9              Dragons in the Sunset        Fire + Ice   
4      E2DC1FAE                        Soul People        Space Echo   
...         ...                                ...               ...   
61248  729CBB09                            My Name            McLean   
61249  D08D4A55  Maybe One Day (feat. Black Spade)       Blu & Exile   
61250  C5E3A0D5                          Jalopiina           unknown   
61251  321D0506                      Freight Train     Chas McDevitt   
61252  3A64EF84          Tell Me Sweet Little Lies      Monica Lopez   

            genre              city      time        day  
0            rock  Saint-Petersburg  20:28:33  Wednesday  
1            rock

In [15]:
df.duplicated().sum()

0

Unique genre names in the dataset:

In [17]:
genre = df['genre']
genre = genre.sort_values()
print(genre.unique())

['acid' 'acoustic' 'action' 'adult' 'africa' 'afrikaans' 'alternative'
 'alternativepunk' 'ambient' 'americana' 'animated' 'anime' 'arabesk'
 'arabic' 'arena' 'argentinetango' 'art' 'audiobook' 'author' 'avantgarde'
 'axé' 'baile' 'balkan' 'beats' 'bigroom' 'black' 'bluegrass' 'blues'
 'bollywood' 'bossa' 'brazilian' 'breakbeat' 'breaks' 'broadway'
 'cantautori' 'cantopop' 'canzone' 'caribbean' 'caucasian' 'celtic'
 'chamber' 'chanson' 'children' 'chill' 'chinese' 'choral' 'christian'
 'christmas' 'classical' 'classicmetal' 'club' 'colombian' 'comedy'
 'conjazz' 'contemporary' 'country' 'cuban' 'dance' 'dancehall' 'dancepop'
 'dark' 'death' 'deep' 'deutschrock' 'deutschspr' 'dirty' 'disco' 'dnb'
 'documentary' 'downbeat' 'downtempo' 'drum' 'dub' 'dubstep' 'eastern'
 'easy' 'electronic' 'electropop' 'emo' 'entehno' 'epicmetal' 'estrada'
 'ethnic' 'eurofolk' 'european' 'experimental' 'extrememetal' 'fado'
 'fairytail' 'film' 'fitness' 'flamenco' 'folk' 'folklore' 'folkmetal'
 'folkrock' 

We remove implicit duplicates in the genre names

In [19]:
duplicates = ['hip', 'hop', 'hip-hop']  # list of incorrect genre names
name = 'hiphop'  # correct genre name
df['genre'] = df['genre'].replace(duplicates, name)  # replace all values from duplicates with name
# print(df['genre'])  # the dataframe has been updated, and implicit duplicates have been removed

In [20]:
genre = df['genre']
genre = genre.sort_values()
print(genre.unique())

['acid' 'acoustic' 'action' 'adult' 'africa' 'afrikaans' 'alternative'
 'alternativepunk' 'ambient' 'americana' 'animated' 'anime' 'arabesk'
 'arabic' 'arena' 'argentinetango' 'art' 'audiobook' 'author' 'avantgarde'
 'axé' 'baile' 'balkan' 'beats' 'bigroom' 'black' 'bluegrass' 'blues'
 'bollywood' 'bossa' 'brazilian' 'breakbeat' 'breaks' 'broadway'
 'cantautori' 'cantopop' 'canzone' 'caribbean' 'caucasian' 'celtic'
 'chamber' 'chanson' 'children' 'chill' 'chinese' 'choral' 'christian'
 'christmas' 'classical' 'classicmetal' 'club' 'colombian' 'comedy'
 'conjazz' 'contemporary' 'country' 'cuban' 'dance' 'dancehall' 'dancepop'
 'dark' 'death' 'deep' 'deutschrock' 'deutschspr' 'dirty' 'disco' 'dnb'
 'documentary' 'downbeat' 'downtempo' 'drum' 'dub' 'dubstep' 'eastern'
 'easy' 'electronic' 'electropop' 'emo' 'entehno' 'epicmetal' 'estrada'
 'ethnic' 'eurofolk' 'european' 'experimental' 'extrememetal' 'fado'
 'fairytail' 'film' 'fitness' 'flamenco' 'folk' 'folklore' 'folkmetal'
 'folkrock' 

**Conclusions**

Data preprocessing revealed three issues in the dataset:

- inconsistent column naming style,
- missing values,
- duplicates, both explicit and implicit.

We corrected the column names to make the table easier to work with. Removing duplicates also makes the analysis more accurate.

We replaced missing values with `'unknown'`. It remains to be seen whether the missing values in the `genre` column will affect the results of the study.

Now we can move on to hypothesis testing.


## 2. Hypothesis Testing

### 2.1 Comparing User Behavior in Two Cities

The first hypothesis states that users in Moscow and Saint Petersburg listen to music differently. We test this assumption using data for three days of the week: Monday, Wednesday, and Friday.

To do this, we:

- separate users from Moscow and Saint Petersburg,
- compare how many tracks each group listened to on Monday, Wednesday, and Friday.


The number of tracks played in each city:

In [21]:
df_grouping = df.groupby('city')['time'].count()
print(df_grouping)

city
Moscow              42741
Saint-Petersburg    18512
Name: time, dtype: int64


We group the data by day of the week and count the number of plays on Monday, Wednesday, and Friday.

In [22]:
day_grouping = df.groupby('day')['time'].count()
print(day_grouping)

day
Friday       21840
Monday       21354
Wednesday    18059
Name: time, dtype: int64


We define the `number_tracks()` function with two parameters: `day` and `city`.

In [24]:
def number_tracks (day, city):
    track_list = df[(df['day'] == day) & (df['city'] == city)]
    track_list_count = track_list['user_id'].count()
    return track_list_count # возвращаем значение

In [25]:
msk1 = number_tracks('Monday', 'Moscow')
print(f'Number of tracks on Monday in Moscow: {msk1}')

msk2 = number_tracks('Wednesday', 'Moscow')
print(f'Number of tracks on Wednesday in Moscow: {msk2}')

msk3 = number_tracks('Friday', 'Moscow')
print(f'Number of tracks on Friday in Moscow: {msk3}')
print()

spb1 = number_tracks('Monday', 'Saint-Petersburg')
print(f'Number of tracks on Monday in Saint-Petersburg: {spb1}')

spb2 = number_tracks('Wednesday', 'Saint-Petersburg')
print(f'Number of tracks on Wednesday in Saint-Petersburg: {spb2}')

spb3 = number_tracks('Friday', 'Saint-Petersburg')
print(f'Number of tracks on Friday in Saint-Petersburg: {spb3}')

Number of tracks on Monday in Moscow: 15740
Number of tracks on Wednesday in Moscow: 11056
Number of tracks on Friday in Moscow: 15945

Number of tracks on Monday in Saint-Petersburg: 5614
Number of tracks on Wednesday in Saint-Petersburg: 7003
Number of tracks on Friday in Saint-Petersburg: 5895


**Table with the results**

* column names: `['city', 'monday', 'wednesday', 'friday']`
* data: the results obtained using `number_tracks()`

In [29]:
columns = ['city', 'Monday', 'Wednesday', 'Friday']

info = pd.DataFrame(
    data=[
        ['Moscow', msk1, msk2, msk3],
        ['Saint Petersburg', spb1, spb2, spb3]
    ],
    columns=columns
)
info

,city,Monday,Wednesday,Friday
0,Moscow,15740,11056,15945
1,Saint Petersburg,5614,7003,5895


### 2.2 Music at the Beginning and End of the Week

According to the second hypothesis, different genres dominate in Moscow and Saint Petersburg on Monday mornings. The same is expected to be true on Friday evenings, when genre preferences should also differ depending on the city.

We create the `moscow_general` table by selecting the rows from `df` where the value in the `city` column is `'Moscow'`

In [30]:
moscow_general = df[(df['city'] == 'Moscow')]
moscow_general

,user_id,track,artist,genre,city,time,day
1,55204538,Delayed Because of Accident,Andreas Rönnberg,rock,Moscow,14:07:09,Friday
4,E2DC1FAE,Soul People,Space Echo,dance,Moscow,08:34:34,Monday
6,4CB90AA5,True,Roman Messer,dance,Moscow,13:00:07,Wednesday
7,F03E1C1F,Feeling This Way,Polina Griffith,dance,Moscow,20:47:49,Wednesday
8,8FA1D3BE,И вновь продолжается бой,unknown,ruspop,Moscow,09:17:40,Friday
...,...,...,...,...,...,...,...
61247,83A474E7,I Worship Only What You Bleed,The Black Dahlia Murder,extrememetal,Moscow,21:07:12,Monday
61248,729CBB09,My Name,McLean,rnb,Moscow,13:32:28,Wednesday
61250,C5E3A0D5,Jalopiina,unknown,industrial,Moscow,20:09:26,Friday
61251,321D0506,Freight Train,Chas McDevitt,rock,Moscow,21:43:59,Friday


We create the `spb_general` table by selecting the rows from `df` where the value in the `city` column is `'Saint Petersburg'`

In [31]:
spb_general = df[(df['city'] == 'Saint-Petersburg')]
spb_general

,user_id,track,artist,genre,city,time,day
0,FFB692EC,Kamigata To Boots,The Mass Missile,rock,Saint-Petersburg,20:28:33,Wednesday
2,20EC38,Funiculì funiculà,Mario Lanza,pop,Saint-Petersburg,20:58:07,Wednesday
3,A3DD03C9,Dragons in the Sunset,Fire + Ice,folk,Saint-Petersburg,08:37:09,Monday
5,842029A1,Преданная,IMPERVTOR,rusrap,Saint-Petersburg,13:09:41,Friday
9,E772D5C0,Pessimist,unknown,dance,Saint-Petersburg,21:20:49,Wednesday
...,...,...,...,...,...,...,...
61239,D94F810B,Theme from the Walking Dead,Proyecto Halloween,film,Saint-Petersburg,21:14:40,Monday
61240,BC8EC5CF,Red Lips: Gta (Rover Rework),Rover,electronic,Saint-Petersburg,21:06:50,Monday
61241,29E04611,Bre Petrunko,Perunika Trio,world,Saint-Petersburg,13:56:00,Monday
61242,1B91C621,(Hello) Cloud Mountain,sleepmakeswaves,postrock,Saint-Petersburg,09:22:13,Monday


We define the `genre_weekday()` function with the parameters `table`, `day`, `time1`, and `time2` to identify the most popular genres for a selected day and time interval.

Inside the function:

- We store in `genre_df` only those rows from the input dataframe where:
  - the value in the `day` column matches the `day` argument,
  - the value in the `time` column is greater than `time1`,
  - the value in the `time` column is less than `time2`.

- Then we group `genre_df` by the `genre` column, select one column, and use the `count()` method to calculate the number of records for each genre. The resulting Series is stored in `genre_df_count`.

- Next, we sort `genre_df_count` in descending order of frequency and store the result in `genre_df_sorted`.

- Finally, the function returns the first 10 values of `genre_df_sorted`, which represent the top 10 most popular genres for the specified day and time range.


In [46]:
def genre_weekday(df, day, time1, time2):
    # consecutive filtering
    # keep only the rows where the day is equal to day
    genre_df = df[df['day'] == day]

    # keep only the rows where the time is earlier than time2
    genre_df = genre_df[genre_df['time'] < time2]

    # keep only the rows where the time is later than time1
    genre_df = genre_df[genre_df['time'] > time1]

    # group the filtered dataframe by genre and count the number of rows for each genre
    genre_df_grouped = genre_df.groupby('genre')['genre'].count()

    # sort the result in descending order so that the most popular genres appear first
    genre_df_sorted = genre_df_grouped.sort_values(ascending=False)

    # return the top 10 most popular genres for the specified time period and day
    return genre_df_sorted[:10]

In [47]:
msk_time1 = genre_weekday(moscow_general, 'Monday', '07:00', '11:00')
print(msk_time1)

genre
pop            781
dance          549
electronic     480
rock           474
hiphop         286
ruspop         186
world          181
rusrap         175
alternative    164
unknown        161
Name: genre, dtype: int64


In [48]:
spb_time1 = genre_weekday(spb_general, 'Monday', '07:00', '11:00')
print(spb_time1)

genre
pop            218
dance          182
rock           162
electronic     147
hiphop          80
ruspop          64
alternative     58
rusrap          55
jazz            44
classical       40
Name: genre, dtype: int64


In [49]:
msk_time2 = genre_weekday(moscow_general, 'Friday', '17:00', '23:00')
print(msk_time2)

genre
pop            713
rock           517
dance          495
electronic     482
hiphop         273
world          208
ruspop         170
alternative    163
classical      163
rusrap         142
Name: genre, dtype: int64


In [50]:
spb_time2 = genre_weekday(spb_general, 'Friday', '17:00', '23:00')
print(spb_time2)

genre
pop            256
electronic     216
rock           216
dance          210
hiphop          97
alternative     63
jazz            61
classical       60
rusrap          59
world           54
Name: genre, dtype: int64


**Conclusions**

If we compare the top 10 genres on Monday morning, we can draw the following conclusions:

1. Users in Moscow and Saint Petersburg listen to largely similar music. The main difference is that the Moscow ranking includes `world`, while the Saint Petersburg ranking includes `jazz` and `classical`.

2. In Moscow, the number of missing values is so high that `'unknown'` ranks tenth among the most popular genres. This means that missing values account for a substantial share of the data and may affect the reliability of the analysis.

Friday evening does not change the overall picture. Some genres move slightly up or down, but in general the top 10 remains nearly the same.

Therefore, the second hypothesis was only partially confirmed:

- Users listen to broadly similar music both at the beginning and at the end of the week.
- The difference between Moscow and Saint Petersburg is not very pronounced. Russian pop music is more common in Moscow, while jazz is more prominent in Saint Petersburg.

However, the missing values cast doubt on this result. In Moscow, they are numerous enough that the top 10 ranking might have looked different if the missing genre data had been available.


### 2.3 Genre Preferences in Moscow and Saint Petersburg

**Hypothesis:** Saint Petersburg is the capital of rap, where this genre is listened to more often than in Moscow. Moscow, on the other hand, is a city of contrasts where pop music still remains the dominant genre.


We group the `moscow_general` table by the `genre` column, count the number of `genre` values in each group using the `count()` method, sort the resulting Series in descending order, and store it in `moscow_genres`.

In [41]:
moscow_genres = moscow_general.groupby('genre')['genre'].count()
moscow_genres = moscow_genres.sort_values(ascending=False)

In [42]:
print(moscow_genres.head(10))

genre
pop            5892
dance          4435
rock           3965
electronic     3786
hiphop         2096
classical      1616
world          1432
alternative    1379
ruspop         1372
rusrap         1161
Name: genre, dtype: int64


We group the `spb_general` table by the `genre` column, count the number of `genre` values in each group using the `count()` method, sort the resulting Series in descending order, and store it in `spb_genres`.

In [43]:
spb_genres = spb_general.groupby('genre')['genre'].count()
spb_genres = spb_genres.sort_values(ascending=False)

In [45]:
print(spb_genres.head(10))

genre
pop            2431
dance          1932
rock           1879
electronic     1736
hiphop          960
alternative     649
classical       646
rusrap          564
ruspop          538
world           515
Name: genre, dtype: int64


### The hypothesis was partially confirmed:

- Pop music is the most popular genre in Moscow, as suggested in the hypothesis.
- In addition, a closely related genre, Russian pop, also appears in the top 10.
- Contrary to expectations, rap is equally popular in both Moscow and Saint Petersburg.

## 4. Final Conclusions

We tested three hypotheses and found the following:

- The day of the week affects user activity differently in Moscow and Saint Petersburg.
  The first hypothesis was fully confirmed.

- Music preferences do not change significantly over the course of the week in either Moscow or Saint Petersburg. However, some differences can be seen at the beginning of the week, on Mondays:
  - users in Moscow tend to listen more to `world` music,
  - users in Saint Petersburg prefer `jazz` and `classical`.

- Therefore, the second hypothesis was only partially confirmed. The result might have been different if the dataset had contained fewer missing values.

- The musical tastes of users in Moscow and Saint Petersburg have more in common than differences. Contrary to expectations, genre preferences in Saint Petersburg are quite similar to those in Moscow.

- The third hypothesis was not confirmed. Even if differences in preferences do exist, they are not clearly visible across the majority of users.